# Task 3 — A/B hypothesis testing (ACIS)

**Objective:** Statistically validate or reject hypotheses on **claim frequency**, **claim severity**, and **margin** to support segmentation and pricing.

| Hypothesis | KPI | Test |
|------------|-----|------|
| No provincial risk difference | Claim frequency (policies with ≥1 claim) | Chi-squared |
| No zip-code risk difference (matched cohort) | Claim frequency | Chi-squared |
| No zip-code margin difference | Margin = ΣPremium − ΣClaims per policy | Welch *t* |
| No gender risk difference | Claim frequency | Chi-squared |

**Matched cohort for postal codes:** same `Province` and `VehicleType` (default: Gauteng, Passenger Vehicle) so differences are less confounded by vehicle mix, then compare two high-volume postal codes (defaults: `2000` vs `122`).

Reusable functions live in `src/hypothesis_tests.py`. A written summary is in `reports/hypothesis_testing_results.md`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import load_insurance_data
from src.hypothesis_tests import (
    aggregate_policy_kpis,
    business_interpretation,
    results_to_dataframe,
    run_hypothesis_suite,
)

df = load_insurance_data()
print(df.shape)
df.head(2)

In [ ]:
policies = aggregate_policy_kpis(df)
print("Policies:", len(policies))
policies.head()

In [ ]:
ALPHA = 0.05
results = run_hypothesis_suite(df, alpha=ALPHA)
table = results_to_dataframe(results)
table

In [ ]:
summary = table.assign(
    decision=table["reject_h0"].map({True: "Reject H₀", False: "Fail to reject H₀"})
)
display_cols = [
    "hypothesis_id",
    "kpi",
    "test_name",
    "p_value",
    "decision",
    "n_a",
    "n_b",
    "detail",
]
summary[display_cols]

## Business recommendations

For each **rejected** null, ACIS should consider pricing or portfolio actions; see `reports/hypothesis_testing_results.md` for narrative copy. Below: machine-readable strings.

In [ ]:
for r in results:
    if r.reject_h0:
        print(business_interpretation(r))